# notebook 11 — 本丸 (2)：全結合連続スピン系の直接モンテカルロ（Julia）

**実行環境：MacBook Pro M4 / 24GB / Julia 1.10+。標準ライブラリのみ（外部パッケージ不要）。**

nb10（Python・平均場）の open 1 を引き継ぐ。nb10 で確定したのは、全結合・符号つき `cos2` 結合の
平均場が臨界点 `β*=4N=12` を持つ（判定I＝非コンパクト化の入口クリア）が、破れた相の実効相関は
**rank 2・k=2 の周期構造を保ち**、符号 `sign(cos2θ)` の周期反転は解消しない（判定II＝符号は
平均場最低次では通らない）こと。

nb10 の判定IIは平均場**最低次**。符号を破る非周期構造は、もし存在するなら連結相関の**高次
（ゆらぎ）**に隠れている可能性がある。この notebook はそれを直接モンテカルロで検証する。

**問い**：有限サイズ全結合系を `β*=12` 近傍で MC サンプリングし、測定した**実効相関**および
**熱浴分布**に、平均場では見えなかった**非周期モード**（k≠0,2,4、rank>2、奇数次）が立つか。

- 立てば（β） → open 1 が肯定的に進む。符号解消＝単一時間の道が初めて開く。
- 立たなければ（α） → ゆらぎでも周期構造が保たれ、**open 1（作用面）は no-go 確定**。
  `cos2θ` の外からの追加原理が必須、と結論（引き継ぎ書 E-2）。

**規律（教訓D）**：非周期成分らしきものが出ても、有限サイズ効果・統計ノイズ・離散化の人工物で
ないかを必ず確認する（D-3, D-4）。「出た/出ない/判定不能」を正直に記録（E-1）。

## 11.0 セットアップ

In [ ]:
using LinearAlgebra, Statistics, Random, Printf
BLAS.set_num_threads(Sys.CPU_THREADS)
println("Julia ", VERSION, "  CPU threads=", Sys.CPU_THREADS)
const N = 3            # 与件（QCD 同定値）
const TWOPI = 2π

## 11.1 モデルとモンテカルロ（Metropolis）

全結合・符号つき `cos2` 結合。`n` 個のサイト、各サイトに連続角度 `s_i in [0,2π)`。
エネルギー

H = -(1/n) Σ_{i<j} cos2(s_i - s_j)/(2N) = -(1/(2n·2N))·(Mx² + My² - n),

Mx = Σ cos2 s_i,  My = Σ sin2 s_i.

全結合ゆえ各スピンが感じる場は M=(Mx,My) のみで決まり、1スピン更新は M の差分で O(1)。
1スイープ O(n)。n~2000・nsweep~3万 でも M4 で数分／点。

cos2(s_i-s_j)=cos2s_i cos2s_j + sin2s_i sin2s_j の rank-2 構造（nb05 の代数的事実）から、
動的に関係するのは M の 2 成分だけ。これは与件に内在する構造で、新しい量を入れていない。

In [ ]:
# 全結合 Metropolis。M=(Mx,My) を保持して O(1)/spin 更新。
function mc_core(beta::Float64; n=2000, nsweep=30000, nburn=8000,
                 meas_every=20, seed=1)
    rng = MersenneTwister(seed)
    s  = TWOPI .* rand(rng, n)
    c2 = cos.(2 .* s); s2 = sin.(2 .* s)
    Mx = sum(c2); My = sum(s2)
    coef = beta/(2*n*2N)          # dH = -coef * d(M^2)
    MxA = Float64[]; MyA = Float64[]
    @inbounds for sweep in 1:nsweep
        for _ in 1:n
            i  = rand(rng, 1:n)
            ns_ = TWOPI*rand(rng)
            nc = cos(2ns_); nsi = sin(2ns_)
            dMx = nc - c2[i]; dMy = nsi - s2[i]
            dM2 = (2*Mx*dMx + dMx^2) + (2*My*dMy + dMy^2)
            dH  = -coef * dM2
            if dH <= 0 || rand(rng) < exp(-dH)
                s[i]=ns_; c2[i]=nc; s2[i]=nsi; Mx+=dMx; My+=dMy
            end
        end
        if sweep > nburn && sweep % meas_every == 0
            push!(MxA, Mx/n); push!(MyA, My/n)
        end
    end
    Rs = hypot.(MxA, MyA)
    return (Rmean=mean(Rs), Rstd=std(Rs), mx=mean(MxA), my=mean(MyA))
end

# サニティ：平均場 beta*=12 を MC が再現するか（R が 12 付近で立ち上がる）
println("sanity: R(beta) by MC")
for b in [8.0, 11.0, 12.0, 13.0, 16.0]
    r = mc_core(b; n=1500, nsweep=15000, nburn=5000, seed=1)
    @printf("  beta=%4.1f : R=%.4f (std %.4f)\n", b, r.Rmean, r.Rstd)
end

## 11.2 実効相関と熱浴分布の測定

- **観測量A**：m 個の固定プローブ設定 θ_a が感じる場 h_a=(cos2θ_a,sin2θ_a)·(Mx/n,My/n) の
  連結相関 C_eff(a,b)=⟨h_a h_b⟩−⟨h_a⟩⟨h_b⟩。h_a は M の線形結合ゆえ定義上 rank≤2——これは
  「平均場と同じ土俵」の基準線。
- **観測量B（本命）**：熱浴スピンの角度分布 ρ(s−φ)（磁化方向 φ に揃える）。平均場では
  ρ ∝ exp(βR cos2(s−φ)/2N) で偶数次（周期 π in 2s）のみ。**奇数次 k=1,3 や rank>2 が
  ゆらぎで立てば**、cos2 の周期性を超えた非周期性の兆候。

In [ ]:
function mc_observe(beta::Float64; n=2000, nsweep=30000, nburn=8000,
                    meas_every=20, m=180, seed=1)
    rng = MersenneTwister(seed)
    s  = TWOPI .* rand(rng, n)
    c2 = cos.(2 .* s); s2 = sin.(2 .* s)
    Mx = sum(c2); My = sum(s2)
    coef = beta/(2*n*2N)
    θ  = collect(range(0, TWOPI, length=m+1))[1:m]
    pc = cos.(2 .* θ); ps = sin.(2 .* θ)
    crossA=zeros(m,m); meanA=zeros(m); cntA=0
    hist = zeros(m); dθ = TWOPI/m
    @inbounds for sweep in 1:nsweep
        for _ in 1:n
            i=rand(rng,1:n); ns_=TWOPI*rand(rng); nc=cos(2ns_); nsi=sin(2ns_)
            dMx=nc-c2[i]; dMy=nsi-s2[i]
            dM2=(2*Mx*dMx+dMx^2)+(2*My*dMy+dMy^2); dH=-coef*dM2
            if dH<=0 || rand(rng)<exp(-dH)
                s[i]=ns_; c2[i]=nc; s2[i]=nsi; Mx+=dMx; My+=dMy
            end
        end
        if sweep>nburn && sweep%meas_every==0
            mx=Mx/n; my=My/n
            h = pc.*mx .+ ps.*my
            for a in 1:m
                meanA[a]+=h[a]
                @simd for b in 1:m; crossA[a,b]+=h[a]*h[b]; end
            end
            φ = 0.5*atan(my, mx)
            for j in 1:n
                u = mod(s[j]-φ, TWOPI)
                k = 1 + min(m-1, Int(floor(u/dθ)))
                hist[k]+=1
            end
            cntA+=1
        end
    end
    meanA./=cntA; crossA./=cntA
    Ceff = crossA .- (meanA*meanA')
    hist ./= (cntA*n*dθ)
    return Ceff, hist, θ, cntA
end
println("mc_observe defined")

## 11.3 解析：フーリエ次数とランク

In [ ]:
function dft_lowk(x::Vector{Float64}, K::Int=8)
    m=length(x); out=zeros(K+1)
    for k in 0:K
        re=0.0; im=0.0
        for j in 0:m-1
            ang=-2π*k*j/m; re+=x[j+1]*cos(ang); im+=x[j+1]*sin(ang)
        end
        out[k+1]=hypot(re,im)/m
    end
    out
end

function analyze(Ceff, hist)
    m=size(Ceff,1)
    Cdelta=zeros(m)
    @inbounds for a in 1:m, d in 0:m-1
        Cdelta[d+1]+=Ceff[a, 1+mod(a-1+d,m)]
    end
    Cdelta./=m
    Fc = dft_lowk(Cdelta); Fc./=(maximum(Fc)+eps())
    ev = sort(abs.(eigen(Symmetric(0.5*(Ceff+Ceff'))).values), rev=true)
    rk = count(>(1e-6*ev[1]), ev)
    Fh = dft_lowk(hist); Fh./=(Fh[1]+eps())
    return Cdelta, Fc, ev, rk, Fh
end
println("analyze defined")

In [ ]:
betas = [8.0, 11.0, 12.0, 12.5, 14.0, 18.0]
println("== C_eff フーリエ |F_k| (k=0..8) と rank、ρ(s-φ) フーリエ ==")
for beta in betas
    Ceff, hist, θ, cnt = mc_observe(beta; n=2000, nsweep=30000, nburn=8000, m=180, seed=12345)
    _, Fc, ev, rk, Fh = analyze(Ceff, hist)
    @printf("β=%5.1f rank=%d\n", beta, rk)
    println("   C_eff |F_k| = ", join([@sprintf("%.3f",f) for f in Fc], " "))
    println("   ρ(s-φ)|F_k| = ", join([@sprintf("%.3f",f) for f in Fh], " "))
end

## 11.4 有限サイズチェック（人工物の排除）

臨界点直上 β=12 で n=1000,2000,4000 と変え、非周期成分（ρ の k=1,3、C_eff の rank>2 / k≠2）が
n→∞ で残るかを見る。残れば本物、1/√n で消えれば統計ノイズ／有限サイズ。

In [ ]:
println("== FSS @ β=12.0 ==")
for n in [1000, 2000, 4000]
    Ceff, hist, θ, cnt = mc_observe(12.0; n=n, nsweep=40000, nburn=10000, m=180, seed=777)
    _, Fc, ev, rk, Fh = analyze(Ceff, hist)
    @printf("n=%4d : rank=%d  ρ:k1=%.4f k3=%.4f  C:k1=%.4f k3=%.4f  (ρ:k2=%.3f C:k2=%.3f)\n",
            n, rk, Fh[2], Fh[4], Fc[2], Fc[4], Fh[3], Fc[3])
end

## 11.5 サマリー（MC 実行後に記入）

> 想定される二分岐（教訓D：FSS で人工物を排除してから結論）：
>
> **(α) 非周期モードが立たない** — ρ(s−φ) が偶数次（k=2,4,…）のみ、C_eff が rank≤2・k=2 のみ、
> 非周期成分が n→∞ で 1/√n 減衰 → ゆらぎでも cos2θ の周期構造は保たれる。
> **open 1（作用面）は no-go 確定。** 目標 (3+1) 時空には cos2θ の外からの追加原理が必須。
> 引き継ぎ書 E-2 の議論（追加原理として何を選ぶか）へ移行。
>
> **(β) 非周期モードが立つ** — ρ に k=1,3 等の奇数次、または C_eff に rank>2／k≠2 が n→∞ で
> **残存** → ゆらぎが周期性を破る。open 1 が初めて肯定的に進む。次：破れた相での MDS 復元幾何
> （円が壊れ非コンパクト方向が開くか）と、符号 sign の周期反転が解消し単一時間が出るか（判定II）。
>
> ### established/open（暫定）
> - established：全結合系の直接 MC を実装し、臨界点近傍で実効相関・熱浴分布のフーリエ／ランク／
>   有限サイズ依存を測定できる。サニティで平均場 β*=12 を MC が再現することを確認。
> - open：(α)/(β) の判定は実行結果による。結果と判断を本セルに追記すること。

### 次 Chat への申し送り
- 本 notebook 実行 → (α)/(β) を判定 → 引き継ぎ書 C.open1 を解決済みに更新。
- (α) なら no-go が作用面で確定し、引き継ぎ書の主結論（追加原理が論理的に必要）が
  st・幾何・作用の全面で閉じる。(β) なら判定II（符号・単一時間）へ進む新 notebook を起こす。

### 予備検証（Python 等価実装、small run）の知見 — 2026-06
- ロジック検証済み：mc_core/mc_observe/analyze は正常動作。サニティで R(beta) が beta~12 で
  立ち上がり、平均場 beta*=12 を MC が再現。
- 予備結果は **(α) 方向を強く示唆**：熱浴分布 rho(s-phi) のフーリエは **偶数次 k=2,4,6 のみ**
  （beta=16 で k2=0.645, k4=0.249, k6=0.067）、**奇数次 k=1,3,5,7 はノイズレベル(<0.002)**。
  C_eff は全 beta で rank2・k=2 のみ。ゆらぎを入れても非周期(奇数次)モードは立っていない。
- ただし small run (n=1000, nsweep=8000)。**M4 本実行（n=2000-4000, 長スイープ）と臨界点直上の
  FSS（奇数次が n→∞ で残るか）で (α) を確定すること。** 教訓D-4：予備結果に飛びつかない。

### 本番結果（M4 / Julia）— (α) 確定  2026-06
β走査：C_eff は全βで rank2・k=2 のみ。ρ(s-φ) は偶数次 k=2,4,6 のみ階層的に成長
（β=18: k2=0.723,k4=0.332,k6=0.110）、**奇数次 k=1,3,5,7 は全βでノイズ床(<=0.005)**。
FSS@β=12：n=1000→2000→4000 で ρ:k1≈0.0005, k3≈0.0007〜0.0010 と **n 非依存・床のまま**、
C の奇数次は厳密に 0、rank=2 不変。k2 は 0.176→0.148→0.123 と有限サイズ寄与が正しく減衰。
→ 非周期(奇数次)モードは人工物ですらなく、立っていない。**判定 (α) 確定。**

**結論：open 1（本丸）は no-go として閉じた。** ゆらぎを全次数含む直接 MC でも cos2θ の
周期構造は破れず、sign(cos2θ) の周期反転を解消する非周期モードは出ない。引き継ぎ書 A-6 の
no-go が st・幾何・作用の全面で確定。目標 (3+1) 時空には cos2θ の外からの追加原理が論理的に必須。
次：引き継ぎ書 E-2（追加原理の選定）へ。